# 🕵️ SQL JOIN 탐정단: 학교 간식 가게
## 두 장부를 연결해 간식 가게의 비밀을 찾아라!

> 학생용 실습 노트북 · TODO + 자동점검

오늘 해결할 질문:
1. 누가 어떤 간식을 샀을까?
2. 구매하지 않은 친구는 누구일까?
3. 주인이 없는 주문도 찾을 수 있을까?
4. 누가 가장 많이 샀을까?

> **저장할 때는 나누고, 찾을 때는 JOIN으로 연결한다.**


# 1. 오늘의 학습 목표

- 테이블을 나누는 이유를 설명할 수 있다.
- `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, `FULL OUTER JOIN`의 차이를 이해한다.
- `ON` 조건으로 두 표를 연결할 수 있다.
- `IS NULL`로 연결되지 않은 데이터를 찾을 수 있다.
- `GROUP BY`, `COUNT`, `SUM`을 JOIN과 함께 사용할 수 있다.

### 가장 쉬운 기억법
- INNER JOIN → **짝이 있는 것만**
- LEFT JOIN → **왼쪽은 모두**
- RIGHT JOIN → **오른쪽은 모두**
- FULL OUTER JOIN → **양쪽 모두**


# 2. 실습 환경 준비

이번 실습은 Google Colab의 Python + SQLite를 사용합니다.

> 아래 셀은 수정하지 말고 실행하세요.


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

def sql(query):
    return pd.read_sql_query(query, conn)

print("✅ SQLite 준비 완료")
print("SQLite version:", sqlite3.sqlite_version)


# 3. 왜 표를 두 개로 나눌까요?

주문할 때마다 이름까지 계속 적으면 같은 정보가 반복됩니다.

| 주문 | 회원번호 | 이름 | 간식 |
|---:|---:|---|---|
| 1 | 101 | 지민 | 초코우유 |
| 2 | 101 | 지민 | 쿠키 |
| 3 | 101 | 지민 | 젤리 |

지민이가 이름을 바꾸면 여러 줄을 모두 고쳐야 합니다.

그래서:

- 사람 정보 → `Users`
- 주문 정보 → `Orders`

로 나눕니다.

> **나누는 이유 = 반복 줄이기**  
> **JOIN하는 이유 = 나눈 정보를 다시 연결해서 답 찾기**


# 4. 실습 데이터 만들기

### Users
| id | name | grade |
|---:|---|---:|
|101|지민|4|
|102|민수|5|
|103|서아|4|
|104|준호|6|
|105|하린|5|

### Orders
| order_id | user_id | item | price |
|---:|---:|---|---:|
|1|101|초코우유|1500|
|2|101|쿠키|1000|
|3|102|젤리|2000|
|4|104|빵|2500|
|5|NULL|사과주스|1800|

`NULL`은 0이 아니라 **기록된 값이 없음**입니다.


In [ ]:
conn.executescript("""
CREATE TABLE Users(
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    grade INTEGER NOT NULL
);

CREATE TABLE Orders(
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    item TEXT NOT NULL,
    price INTEGER NOT NULL
);

INSERT INTO Users VALUES
(101,'지민',4),
(102,'민수',5),
(103,'서아',4),
(104,'준호',6),
(105,'하린',5);

INSERT INTO Orders VALUES
(1,101,'초코우유',1500),
(2,101,'쿠키',1000),
(3,102,'젤리',2000),
(4,104,'빵',2500),
(5,NULL,'사과주스',1800);
""")

display(sql("SELECT * FROM Users ORDER BY id"))
display(sql("SELECT * FROM Orders ORDER BY order_id"))


# 5. SQL 자동점검 도우미

기존 SQL 실습 파일처럼 TODO를 완성한 뒤 자동점검합니다.

> 아래 셀은 수정하지 마세요.


In [ ]:
TODO_STATUS = {}

def run_sql(q):
    if "TODO" in q:
        print("⚠️ TODO가 남아 있습니다.")
        return
    try:
        display(sql(q))
        print("✅ 실행 완료")
    except Exception as e:
        print("❌ SQL 오류:", e)

def check_todo(n, student_sql, answer_sql):
    if "TODO" in student_sql:
        TODO_STATUS[n] = False
        print(f"TODO {n}: ❌ 먼저 TODO를 완성하세요.")
        return False
    try:
        a = sql(student_sql)
        b = sql(answer_sql)
        ok = list(a.columns)==list(b.columns) and a.fillna("<NULL>").astype(str).equals(b.fillna("<NULL>").astype(str))
    except Exception as e:
        TODO_STATUS[n]=False
        print("❌ SQL 오류:", e)
        return False
    TODO_STATUS[n]=ok
    print(f"TODO {n}: " + ("✅ 통과" if ok else "❌ 다시 확인"))
    if not ok:
        print("예상 결과")
        display(b)
        print("현재 결과")
        display(a)
    return ok

def show_progress(total):
    for i in range(1,total+1):
        s=TODO_STATUS.get(i)
        print(("✅" if s else "❌" if s is False else "⬜"), f"TODO {i}")


# 6. JOIN SQL 읽는 법

```sql
SELECT U.name, O.item
FROM Users AS U
INNER JOIN Orders AS O
    ON U.id = O.user_id;
```

- `SELECT` → 무엇을 볼까?
- `FROM` → 첫 번째 표는?
- `JOIN` → 어느 표를 연결?
- `ON` → 어떤 번호가 같아야 하지?

`U`는 `Users`의 별명, `O`는 `Orders`의 별명입니다.


# TODO 1. INNER JOIN
## 구매한 회원과 간식 찾기

`INNER JOIN`은 **양쪽에서 짝이 있는 행만** 남깁니다.


In [ ]:
student_sql = '''
SELECT
    TODO AS 이름,
    TODO AS 간식
FROM TODO AS U
INNER JOIN TODO AS O
    ON TODO
ORDER BY O.order_id;
'''
run_sql(student_sql)


In [ ]:
check_todo(
1,
student_sql,
'''SELECT U.name AS 이름, O.item AS 간식
FROM Users U
INNER JOIN Orders O ON U.id = O.user_id
ORDER BY O.order_id;'''
)


# TODO 2. 지민이가 산 간식만 찾기

힌트: `WHERE U.name = '지민'`


In [ ]:
student_sql = '''
SELECT U.name AS 이름, O.item AS 간식
FROM Users U
INNER JOIN Orders O ON U.id = O.user_id
WHERE TODO
ORDER BY O.order_id;
'''
run_sql(student_sql)


In [ ]:
check_todo(
2,
student_sql,
'''SELECT U.name AS 이름, O.item AS 간식
FROM Users U
INNER JOIN Orders O ON U.id = O.user_id
WHERE U.name = '지민'
ORDER BY O.order_id;'''
)


# 7. LEFT JOIN
## 왼쪽 회원 명단은 모두 남는다

주문이 없는 서아와 하린도 결과에 남고, 간식은 `NULL`이 됩니다.


# TODO 3. 모든 회원과 주문 보기

In [ ]:
student_sql = '''
SELECT U.name AS 이름, O.item AS 간식
FROM TODO U
TODO JOIN TODO O
    ON TODO
ORDER BY U.id, O.order_id;
'''
run_sql(student_sql)


In [ ]:
check_todo(
3,
student_sql,
'''SELECT U.name AS 이름, O.item AS 간식
FROM Users U
LEFT JOIN Orders O ON U.id = O.user_id
ORDER BY U.id, O.order_id;'''
)


# TODO 4. 주문하지 않은 친구 찾기

힌트: `LEFT JOIN` 후 주문번호가 없는 사람만 찾습니다.

`= NULL`이 아니라 `IS NULL`을 사용합니다.


In [ ]:
student_sql = '''
SELECT U.name AS 이름
FROM Users U
LEFT JOIN Orders O ON U.id = O.user_id
WHERE TODO
ORDER BY U.id;
'''
run_sql(student_sql)


In [ ]:
check_todo(
4,
student_sql,
'''SELECT U.name AS 이름
FROM Users U
LEFT JOIN Orders O ON U.id = O.user_id
WHERE O.order_id IS NULL
ORDER BY U.id;'''
)


# 8. RIGHT JOIN
## 오른쪽 주문 장부를 모두 남긴다

회원번호가 없는 `사과주스` 주문도 결과에 남습니다.


# TODO 5. 모든 주문과 회원 이름 보기

In [ ]:
student_sql = '''
SELECT U.name AS 이름, O.item AS 간식
FROM Users U
TODO JOIN Orders O
    ON TODO
ORDER BY O.order_id;
'''
run_sql(student_sql)


In [ ]:
check_todo(
5,
student_sql,
'''SELECT U.name AS 이름, O.item AS 간식
FROM Users U
RIGHT JOIN Orders O ON U.id = O.user_id
ORDER BY O.order_id;'''
)


# TODO 6. 주인이 없는 주문 찾기

In [ ]:
student_sql = '''
SELECT O.item AS 간식, O.price AS 가격
FROM Users U
RIGHT JOIN Orders O ON U.id = O.user_id
WHERE TODO;
'''
run_sql(student_sql)


In [ ]:
check_todo(
6,
student_sql,
'''SELECT O.item AS 간식, O.price AS 가격
FROM Users U
RIGHT JOIN Orders O ON U.id = O.user_id
WHERE U.id IS NULL;'''
)


# 9. FULL OUTER JOIN
## 회원도 주문도 모두 남긴다

- 연결된 주문 4행
- 주문 없는 회원 2명
- 회원 없는 주문 1건

총 7행입니다.


# TODO 7. 회원과 주문 모두 보기

In [ ]:
student_sql = '''
SELECT U.name AS 이름, O.item AS 간식
FROM Users U
TODO JOIN Orders O
    ON TODO
ORDER BY COALESCE(U.id,999999), O.order_id;
'''
run_sql(student_sql)


In [ ]:
check_todo(
7,
student_sql,
'''SELECT U.name AS 이름, O.item AS 간식
FROM Users U
FULL OUTER JOIN Orders O ON U.id = O.user_id
ORDER BY COALESCE(U.id,999999), O.order_id;'''
)


# 10. JOIN 네 종류 비교

| JOIN | 남는 기준 | 현재 결과 |
|---|---|---:|
| INNER | 서로 짝이 있는 행만 | 4행 |
| LEFT | 회원은 모두 | 6행 |
| RIGHT | 주문은 모두 | 5행 |
| FULL OUTER | 회원과 주문 모두 | 7행 |

> 이름보다 **어떤 행이 남는지**를 기억하세요.


# 11. 응용: 누가 얼마나 샀을까?

`LEFT JOIN + GROUP BY`를 사용하면 주문하지 않은 회원도 포함한 통계를 만들 수 있습니다.


# TODO 8. 회원별 주문 횟수

In [ ]:
student_sql = '''
SELECT
    U.name AS 이름,
    TODO AS 주문횟수
FROM Users U
LEFT JOIN Orders O ON U.id = O.user_id
GROUP BY TODO, TODO
ORDER BY U.id;
'''
run_sql(student_sql)


In [ ]:
check_todo(
8,
student_sql,
'''SELECT U.name AS 이름,
COUNT(O.order_id) AS 주문횟수
FROM Users U
LEFT JOIN Orders O ON U.id = O.user_id
GROUP BY U.id, U.name
ORDER BY U.id;'''
)


# TODO 9. 회원별 주문 횟수와 총금액

주문이 없는 회원의 합계는 `NULL`이 되므로:

```sql
COALESCE(SUM(O.price), 0)
```

을 사용합니다.


In [ ]:
student_sql = '''
SELECT
    U.name AS 이름,
    TODO AS 주문횟수,
    TODO AS 총금액
FROM Users U
LEFT JOIN Orders O ON U.id = O.user_id
GROUP BY U.id, U.name
ORDER BY U.id;
'''
run_sql(student_sql)


In [ ]:
check_todo(
9,
student_sql,
'''SELECT U.name AS 이름,
COUNT(O.order_id) AS 주문횟수,
COALESCE(SUM(O.price),0) AS 총금액
FROM Users U
LEFT JOIN Orders O ON U.id = O.user_id
GROUP BY U.id, U.name
ORDER BY U.id;'''
)


# TODO 10. 가장 많이 주문한 회원 찾기

In [ ]:
student_sql = '''
SELECT
    U.name AS 이름,
    TODO AS 주문횟수
FROM Users U
LEFT JOIN Orders O ON U.id = O.user_id
GROUP BY U.id, U.name
ORDER BY TODO DESC, U.id ASC
LIMIT TODO;
'''
run_sql(student_sql)


In [ ]:
check_todo(
10,
student_sql,
'''SELECT U.name AS 이름,
COUNT(O.order_id) AS 주문횟수
FROM Users U
LEFT JOIN Orders O ON U.id = O.user_id
GROUP BY U.id, U.name
ORDER BY 주문횟수 DESC, U.id ASC
LIMIT 1;'''
)


# 12. SQL 탐정 미션 3개

## LEVEL 1
지민이가 산 간식 2개 찾기

## LEVEL 2
한 번도 사지 않은 친구 찾기

## LEVEL 3
회원별 주문 횟수 계산하기

> 먼저 결과를 예상한 뒤 SQL을 실행하세요.


In [ ]:
# LEVEL 1 직접 작성
my_sql = '''
SELECT
    -- 직접 작성
'''
# run_sql(my_sql)


In [ ]:
# LEVEL 2 직접 작성
my_sql = '''
SELECT
    -- 직접 작성
'''
# run_sql(my_sql)


In [ ]:
# LEVEL 3 직접 작성
my_sql = '''
SELECT
    -- 직접 작성
'''
# run_sql(my_sql)


# 13. 자주 발생하는 실수

### 1) ON 조건을 잘못 연결
```sql
ON U.id = O.user_id
```

### 2) NULL을 `= NULL`로 비교
잘못된 예:
```sql
WHERE O.order_id = NULL
```

올바른 예:
```sql
WHERE O.order_id IS NULL
```

### 3) LEFT JOIN에서 테이블 순서를 무시
LEFT JOIN은 **왼쪽 테이블이 무엇인지**가 중요합니다.

### 4) 주문 횟수를 `COUNT(*)`로 세기
주문이 없는 회원도 LEFT JOIN 결과에 한 행 남으므로:

```sql
COUNT(O.order_id)
```

로 세는 것이 적절합니다.

### 5) 한 회원은 JOIN 결과에 한 번만 나온다고 생각
지민은 주문을 두 번 했으므로 두 행으로 나타납니다.


# 14. 도전 문제

1. 4학년 학생이 구매한 간식만 출력하세요.
2. 학년별 주문 횟수를 구하세요.
3. 주문하지 않은 학생의 이름과 학년을 출력하세요.
4. 회원이 연결되지 않은 주문의 간식명과 가격을 출력하세요.
5. 총 구매 금액이 가장 큰 회원을 찾아보세요.


# 15. 마지막 퀴즈

1. 이름을 주문 장부에 계속 반복해서 쓰면 어떤 문제가 생길까요?
2. 두 표에서 같은 회원번호를 연결하는 명령은 무엇일까요?
3. INNER JOIN에서 주문 없는 회원은 보일까요?
4. LEFT JOIN에서 주문하지 않은 회원은 보일까요?
5. RIGHT JOIN에서는 회원번호 없는 주문도 보일까요?
6. NULL은 숫자 0과 같은 뜻일까요?
7. 주문 없는 친구를 찾는 조건은 무엇일까요?


# 16. 오늘 배운 내용 정리

| 개념 | 핵심 |
|---|---|
| 테이블 분리 | 반복되는 정보 줄이기 |
| JOIN | 공통 번호로 두 표 연결 |
| INNER JOIN | 양쪽 모두 짝이 있는 행 |
| LEFT JOIN | 왼쪽 모두 |
| RIGHT JOIN | 오른쪽 모두 |
| FULL OUTER JOIN | 양쪽 모두 |
| NULL | 기록된 값 없음 |
| IS NULL | 값 없는 행 찾기 |
| GROUP BY | 같은 기준끼리 묶기 |
| COUNT | 개수 |
| SUM | 합계 |
| COALESCE | NULL을 다른 값으로 표시 |

> **저장할 때는 나누고, 찾을 때는 회원번호로 JOIN한다.**


# 17. TODO 진행 현황

In [ ]:
show_progress(10)

# 18. 자기 점검

- [ ] JOIN이 필요한 이유를 설명할 수 있다.
- [ ] INNER JOIN을 작성할 수 있다.
- [ ] LEFT JOIN을 작성할 수 있다.
- [ ] RIGHT JOIN을 작성할 수 있다.
- [ ] FULL OUTER JOIN을 작성할 수 있다.
- [ ] IS NULL을 사용할 수 있다.
- [ ] 회원별 주문 횟수를 계산할 수 있다.
- [ ] 회원별 총 구매 금액을 계산할 수 있다.
- [ ] JOIN 종류별로 어떤 행이 남는지 설명할 수 있다.


# 19. 👨‍🏫 교사용 90분 운영 가이드

| 시간 | 단계 | 활동 |
|---:|---|---|
| 10분 | 도입 | 이름 반복 저장의 문제 발견 |
| 15분 | 해결 | 표 분리 이유와 회원번호 이해 |
| 10분 | 데이터 | Users / Orders 읽기 |
| 25분 | JOIN | INNER / LEFT 예측 후 실행 |
| 10분 | 확장 | RIGHT / FULL 비교 |
| 15분 | 응용 | 주문 횟수·총금액 미션 |
| 5분 | 정리 | 퀴즈와 한 문장 설명 |

### 공개수업 포인트
SQL을 바로 실행하기 전에 학생에게 먼저 묻습니다.

> **“이 SQL을 실행하면 몇 행이 나올까?”**

결과를 예측하고 확인하게 하면 JOIN의 의미를 더 쉽게 이해할 수 있습니다.
